# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset, analyzed via a Croissant schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL to the Croissant JSON-LD file
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# The metadata object gives direct access to fields like name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values so you can reference them in further exploration.

In [ ]:
# List all record sets with their @id

print("Record sets available in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# --- User Step ---
# From the overview above, pick a record set @id (e.g. 'cr:RecordSet-1'; for demonstration we choose first one)
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

print("RecordSet @ids used to extract data:")
for rsid in record_sets_ids:
    print(f"- {rsid}")

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records from RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} records and {df.shape[1]} columns from {record_set_id}")
        print(f"Column names: {df.columns.tolist()}")
    else:
        print(f"No records found for {record_set_id}")

# For demonstration, pick the first non-empty dataframe
for rid in dataframes.keys():
    break

print("\nPreview of the first few records:")
dataframes[rid].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, normalizing numeric data, and grouping by attributes using the record set and field `@id` identifiers.

In [ ]:
from typing import List
# --- Choose a numeric field by @id (name) from the loaded DataFrame columns ---
# Identify a numeric column automatically for demonstration

df = dataframes[rid]

numeric_fields: List[str] = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Numeric field chosen for analysis: {numeric_field}")
else:
    # Fallback: Try to convert possible numeric columns
    possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ["score","abundance","count","number","percent","ratio","mw","weight"])]
    for col in possible_numeric_fields:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_fields = [col for col in possible_numeric_fields if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Numeric field chosen for analysis: {numeric_field}")
    else:
        print("No suitable numeric field found for EDA.")
        numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # Simple threshold for demo
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to choose a categorical/grouping field
    group_candidates = [col for col in df.columns if (
        col.lower() in ['category', 'sample', 'description', 'type', 'group'] or df[col].dtype == object
    ) and (col != numeric_field) ]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable grouping/categorical field found.")
else:
    print("Skipping EDA as no numeric field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution and grouping if possible
if numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Group-wise distribution (if possible)
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We have successfully loaded the FAIR^2 dataset metadata and records, inspected available record sets, extracted tabular data using record set and field `@id`s via the Croissant specification, and conducted basic exploratory analysis and visualization. Further, domain-specific analyses can continue with this framework.

_For full details on dataset schema elements, documentation, or access to additional record sets, always use the unique `@id` for reference in `mlcroissant` workflows._